# MADA | Fase 1. Tratamento dos logs

Prepara os dados da instanciação empírica da **Metodologia de Amplificação de Divergências Anotativas (MADA)** a partir do corpus MQD e dos logs de anotação da plataforma PCIbex Farm.

**Fluxo:** corpus original → randomização e blocos → extração dos logs → anotações por gênero → consolidação por pluralidade → registro estatístico.

Os arquivos tabulares intermediários são gravados com **tabulação (`\t`)** como separador. A execução sequencial do notebook depende das variáveis definidas nas células anteriores.

## 1. Ambiente e dependências

O notebook identifica execução local (VS Code/Jupyter) ou Google Colab. As bibliotecas são importadas antes da resolução dos caminhos. No Colab, a clonagem de `alexfeitosa72/mada` pressupõe que esse repositório já exista e seja acessível.

**Caminhos centralizados:** `RAIZ`, `DIR_DADOS`, `DIR_BRUTOS` e `DIR_PROCESSADOS`. A execução local pressupõe que o diretório de trabalho seja a raiz de `mada`. A presença de `dados/01-logs-brutos` é verificada antes do processamento.

In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path
import random
import io
import os


import warnings
warnings.filterwarnings('ignore')

In [2]:

from pathlib import Path
import sys
import subprocess

# Identifica o ambiente
IN_COLAB = "google.colab" in sys.modules

REPO = "mada"
URL_REPO = "https://github.com/alexfeitosa72/mada.git"

if IN_COLAB:
    RAIZ = Path("/content") / REPO

    if not RAIZ.exists():
        subprocess.run(
            ["git", "clone", URL_REPO, str(RAIZ)],
            check=True
        )
else:
    RAIZ = Path.cwd().resolve()

# Diretórios do projeto
DIR_DADOS = RAIZ / "dados"
DIR_BRUTOS = DIR_DADOS / "01-logs-brutos"
DIR_PROCESSADOS = DIR_DADOS / "02-logs-processados"

# Confere a pasta de entrada
if not DIR_BRUTOS.is_dir():
    raise FileNotFoundError(
        f"Pasta de entrada não encontrada: {DIR_BRUTOS}"
    )

# Prepara a pasta de saída
DIR_PROCESSADOS.mkdir(parents=True, exist_ok=True)

print("Ambiente:", "Colab" if IN_COLAB else "Local")
print("Raiz:", RAIZ)
print("Entrada:", DIR_BRUTOS)
print("Saída:", DIR_PROCESSADOS)


Ambiente: Local
Raiz: C:\Users\alexander.feitosa\Documents\Github\mada
Entrada: C:\Users\alexander.feitosa\Documents\Github\mada\dados\01-logs-brutos
Saída: C:\Users\alexander.feitosa\Documents\Github\mada\dados\02-logs-processados


## 2. Corpus: randomização e deduplicação

**Entrada:** `DIR_DADOS / "mqd-1465.csv"`.

A semente `SEED = 42` fixa a ordem de embaralhamento. Duplicatas são identificadas pelo texto da coluna `frase`, mantendo-se a primeira ocorrência **na ordem randomizada**; em seguida, as frases são normalizadas conforme o código. A saída `OUTPUT_FILE` é nomeada pela quantidade efetiva de registros e gravada em `DIR_PROCESSADOS`.

**Nota de reprodutibilidade:** a deduplicação ocorre depois da randomização. Alterar essa ordem pode modificar quais IDs originais são preservados.

In [3]:

# ============================================================================
# CONFIGURACAO
# ============================================================================
SEED = 42
INPUT_FILE = DIR_DADOS / "mqd-1465.csv"

print("=" * 80)
print("RANDOMIZACAO MQD-1465 (Seed: 42)")
print("=" * 80)

# ETAPA 1: CARREGAR E RANDOMIZAR
print("\n[1] Carregando e randomizando dados...")

df = pd.read_csv(INPUT_FILE, encoding='utf-8')
df.columns = ['id', 'frase', 'classificacao', 'juizes']
print(f"    {len(df)} registros carregados")

df_random = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"    Randomizado com seed {SEED}")
print(f"    Primeiros IDs: {df_random['id'].head(5).tolist()}")

# ETAPA 2: REMOVER DUPLICATAS
print("\n[2] Removendo duplicatas...")

duplicatas = df_random[df_random.duplicated(subset=['frase'], keep=False)]
if len(duplicatas) > 0:
    print(f"    Duplicatas encontradas:")
    for frase, grupo in duplicatas.groupby('frase'):
        ids = grupo['id'].tolist()
        mantido = df_random[df_random['frase'] == frase]['id'].iloc[0]
        removidos = [i for i in ids if i != mantido]
        print(f"    - IDs {ids}: mantendo {mantido}, removendo {removidos}")

df_final = df_random.drop_duplicates(subset=['frase'], keep='first').reset_index(drop=True)
print(f"    Total apos remocao: {len(df_final)} registros")

# ETAPA 3: PADRONIZAR FRASES
print("\n[3] Padronizando frases...")
df_final['frase'] = df_final['frase'].str.strip().str.replace('"', '', regex=False)
print("    Espacos e aspas removidos")

# ETAPA 4: SALVAR ARQUIVO
num_registros = len(df_final)
OUTPUT_FILE = (
    DIR_PROCESSADOS / f"mqd-{num_registros}-random-sem-repeticao.csv"
)

print(f"\n[4] Salvando '{OUTPUT_FILE}'...")
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# Adiciona aspas nas frases antes de salvar
df_save = df_final.copy()
df_save['frase'] = '"' + df_save['frase'] + '"'
df_save.to_csv(
    OUTPUT_FILE,
    sep='\t',
    index=False,
    encoding='utf-8',
    quoting=csv.QUOTE_NONE,
    escapechar='\\'
)
print("    Arquivo salvo com sucesso!")

# ESTATISTICAS
print("\n" + "=" * 80)
print("ESTATISTICAS")
print("=" * 80)
print(f"\nTotal: {len(df_final)} registros\n")

print("Classificacao:")
for c in sorted(df_final['classificacao'].unique()):
    n = (df_final['classificacao'] == c).sum()
    print(f"  {c:>2}: {n:>4} ({100 * n / len(df_final):>5.1f}%)")

print("\nJuizes:")
for j in sorted(df_final['juizes'].unique()):
    n = (df_final['juizes'] == j).sum()
    print(f"  {j}: {n:>4} ({100 * n / len(df_final):>5.1f}%)")

print("\n" + "=" * 80)
print(f"CONCLUIDO! Arquivo: {OUTPUT_FILE} ({len(df_final)} registros)")
print("=" * 80)


RANDOMIZACAO MQD-1465 (Seed: 42)

[1] Carregando e randomizando dados...
    1465 registros carregados
    Randomizado com seed 42
    Primeiros IDs: [1039, 213, 314, 598, 931]

[2] Removendo duplicatas...
    Duplicatas encontradas:
    - IDs [475, 476]: mantendo 475, removendo [476]
    - IDs [952, 953]: mantendo 952, removendo [953]
    Total apos remocao: 1463 registros

[3] Padronizando frases...
    Espacos e aspas removidos

[4] Salvando 'C:\Users\alexander.feitosa\Documents\Github\mada\dados\02-logs-processados\mqd-1463-random-sem-repeticao.csv'...
    Arquivo salvo com sucesso!

ESTATISTICAS

Total: 1463 registros

Classificacao:
  -1:  544 ( 37.2%)
   0:  409 ( 28.0%)
   1:  510 ( 34.9%)

Juizes:
  2:  525 ( 35.9%)
  3:  936 ( 64.0%)
  4:    2 (  0.1%)

CONCLUIDO! Arquivo: C:\Users\alexander.feitosa\Documents\Github\mada\dados\02-logs-processados\mqd-1463-random-sem-repeticao.csv (1463 registros)


## 3. Identificadores e distribuição em blocos

**Entrada:** `OUTPUT_FILE`, produzido na etapa anterior.

O `id` de origem é preservado em `id_original`; cria-se um identificador sequencial para associar respostas às frases. Os registros são divididos em blocos de até **150 frases**, registrados na coluna `bloco`.

**Saída:** `mqd-<N>-blocos.csv` em `DIR_PROCESSADOS`. O arquivo relaciona `id`, `id_original`, `frase` e `bloco`.

In [4]:

def processar_arquivo_randomizado():
    """
    Processa o arquivo randomizado:
    1. Renomeia id para id_original
    2. Cria novo id sequencial
    3. Distribui os registros em blocos de 150
    """

    # Entrada gerada pela célula 8
    input_path = OUTPUT_FILE

    if not input_path.is_file():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {input_path}"
        )

    print(f"Processando arquivo: {input_path}")

    df = pd.read_csv(
        input_path,
        sep='\t',
        usecols=['id', 'frase'],
        quoting=csv.QUOTE_ALL,
        quotechar='"',
        encoding='utf-8'
    )

    df = df.rename(columns={'id': 'id_original'})
    df.insert(0, 'id', range(1, len(df) + 1))
    df['bloco'] = (df.index // 150) + 1

    # Saída padronizada em kebab-case
    num_registros = len(df)
    output_path = (
        DIR_PROCESSADOS / f"mqd-{num_registros}-blocos.csv"
    )

    df.to_csv(
        output_path,
        sep='\t',
        index=False,
        quoting=csv.QUOTE_ALL,
        quotechar='"'
    )

    print(f"Arquivo salvo em: {output_path}")
    print(
        f"Total de registros: {len(df)} | "
        f"Total de blocos: {df['bloco'].max()}"
    )
    print(f"\nPrimeiras linhas:\n{df.head()}")

    return df


df_blocos_randomizados = processar_arquivo_randomizado()


Processando arquivo: C:\Users\alexander.feitosa\Documents\Github\mada\dados\02-logs-processados\mqd-1463-random-sem-repeticao.csv
Arquivo salvo em: C:\Users\alexander.feitosa\Documents\Github\mada\dados\02-logs-processados\mqd-1463-blocos.csv
Total de registros: 1463 | Total de blocos: 10

Primeiras linhas:
   id  id_original                                              frase  bloco
0   1         1039  Você sabia que o menino que mais vai te dar va...      1
1   2          213  Hoje com 21 anos, como que por auxilio lá do a...      1
2   3          314  Nesse momento to evitando ela, e ta me dando u...      1
3   4          598  Meus sentidos de garota diziam que tinha algum...      1
4   5          931  Aquela promessa que eu te fiz naquele natal, e...      1


## 4. Tratamento dos logs de anotação

**Entradas:** os dez arquivos em `DIR_BRUTOS`, de `results-prod.csv` (bloco 1) a `results-prod-(9).csv` (bloco 10), além do arquivo de blocos da etapa anterior.

O processamento extrai `ParticipantMD5`, gênero autodeclarado (`GeneroCod`), frase, rótulo (`Value`) e duração. A associação depende do ajuste de `ItemNumber` previsto no código para cada bloco; a duração deriva dos eventos `Start` e `End`.

**Saídas:** dez arquivos `bloco-*-concatenado.csv` em `DIR_PROCESSADOS`. O código registra eventuais logs ausentes ou falhas por bloco.

In [5]:
def carregar_blocos_randomizados():
    """Carrega o arquivo com as frases e seus respectivos blocos."""
    arquivos = list(DIR_PROCESSADOS.glob("mqd-*-blocos.csv"))

    if not arquivos:
        raise FileNotFoundError("Nenhum arquivo mqd-*_blocos.csv encontrado!")

    if len(arquivos) > 1:
        arquivos.sort(key=lambda p: p.stat().st_mtime, reverse=True)

    blocos_path = arquivos[0]
    print(f"Carregando arquivo: {blocos_path}")

    return pd.read_csv(
        blocos_path, sep='\t', quoting=csv.QUOTE_ALL,
        quotechar='"', encoding='utf-8'
    )


def extrair_genero(df):
    """Extrai o genero dos participantes do arquivo de log."""
    genero_data = df[
        (df["Label"] == "genero") &
        (df["PennElementName"] == "selecionaGenero") &
        (df["Parameter"] == "Selected")
    ][["ParticipantMD5", "Value"]].copy()

    genero_data["GeneroCod"] = genero_data["Value"].str.lower().map(
        lambda g: "m" if "masculino" in g else "f"
    )

    return genero_data[["ParticipantMD5", "GeneroCod"]]


def calcular_duracoes(df, num_bloco):
    """Calcula duracao entre Start e End para cada trial."""
    df_trials = df[
        (df["Label"] == "frases") &
        (df["Value"].isin(["Start", "End"]))
    ].copy()
    df_trials = df_trials.sort_values(["ParticipantMD5", "ItemNumber", "EventTime"])

    duracoes = []
    for (participant, item), group in df_trials.groupby(["ParticipantMD5", "ItemNumber"]):
        if len(group) != 2:
            continue
        start_time = group[group["Value"] == "Start"]["EventTime"].iloc[0]
        end_time = group[group["Value"] == "End"]["EventTime"].iloc[0]
        adjusted_item = item - 3 if num_bloco == 1 else item - 3 + ((num_bloco - 1) * 150)
        duracoes.append({
            "ParticipantMD5": participant,
            "ItemNumber": adjusted_item,
            "duracao": (end_time - start_time) / 1000.0
        })

    return pd.DataFrame(duracoes)


def processar_arquivo_log(file_path, frases_bloco, num_bloco):
    """Processa um arquivo de log correlacionando frases e calculando duracoes."""
    df = pd.read_csv(file_path, skiprows=19, header=None)
    df.columns = [
        "ReceptionTime", "ParticipantMD5", "Controller", "ItemNumber",
        "InnerElementNumber", "Label", "Group", "PennElementType",
        "PennElementName", "Parameter", "Value", "EventTime", "Comments"
    ]
    df["EventTime"] = pd.to_numeric(df["EventTime"], errors="coerce")

    generos = extrair_genero(df)
    df_duracoes = calcular_duracoes(df, num_bloco)

    df_classificacoes = df[
        (df["Label"] == "frases") &
        (df["Parameter"] == "Selection")
    ].copy()

    if num_bloco == 1:
        df_classificacoes["ItemNumber"] = df_classificacoes["ItemNumber"] - 3
    else:
        df_classificacoes["ItemNumber"] = df_classificacoes["ItemNumber"] - 3 + ((num_bloco - 1) * 150)

    df_final = (
        df_classificacoes
        .merge(frases_bloco[["id", "frase"]], left_on="ItemNumber", right_on="id", how="inner")
        .merge(generos, on="ParticipantMD5", how="left")
        .merge(df_duracoes, on=["ParticipantMD5", "ItemNumber"], how="left")
    )

    return df_final[["ParticipantMD5", "GeneroCod", "frase", "Value", "duracao"]]


def processar_todos_arquivos():
    """Processa todos os arquivos de log, correlacionando com os blocos corretos."""
    pasta_destino = DIR_PROCESSADOS
    os.makedirs(pasta_destino, exist_ok=True)
    df_blocos = carregar_blocos_randomizados()

    for i in range(10):
        num_bloco = i + 1
        arquivo_origem = (
            DIR_BRUTOS / "results-prod.csv"
            if i == 0
            else DIR_BRUTOS / f"results-prod-({i}).csv"
    )
        arquivo_destino = f"{pasta_destino}/bloco-{num_bloco}-concatenado.csv"

        if not arquivo_origem.is_file():
            print(f"Arquivo nao encontrado: {arquivo_origem}")
            continue

        print(f"\nProcessando bloco {num_bloco} - Arquivo: {arquivo_origem}")
        frases_bloco = df_blocos[df_blocos['bloco'] == num_bloco].copy()
        print(f"Frases no bloco {num_bloco}: {len(frases_bloco)}")

        try:
            df_processado = processar_arquivo_log(arquivo_origem, frases_bloco, num_bloco)

            if len(df_processado) > 0:
                df_processado.to_csv(
                    arquivo_destino, sep='\t', index=False,
                    quoting=csv.QUOTE_ALL, quotechar='"',
                    encoding='utf-8', float_format='%.3f'
                )
                print(f"Arquivo salvo: {arquivo_destino} ({len(df_processado)} classificacoes)")
            else:
                print(f"AVISO: Nenhuma classificacao encontrada para o bloco {num_bloco}")
        except Exception as e:
            print(f"Erro ao processar bloco {num_bloco}: {e}")

processar_todos_arquivos()

Carregando arquivo: C:\Users\alexander.feitosa\Documents\Github\mada\dados\02-logs-processados\mqd-1463-blocos.csv

Processando bloco 1 - Arquivo: C:\Users\alexander.feitosa\Documents\Github\mada\dados\01-logs-brutos\results-prod.csv
Frases no bloco 1: 150
Arquivo salvo: C:\Users\alexander.feitosa\Documents\Github\mada\dados\02-logs-processados/bloco-1-concatenado.csv (1350 classificacoes)

Processando bloco 2 - Arquivo: C:\Users\alexander.feitosa\Documents\Github\mada\dados\01-logs-brutos\results-prod-(1).csv
Frases no bloco 2: 150
Arquivo salvo: C:\Users\alexander.feitosa\Documents\Github\mada\dados\02-logs-processados/bloco-2-concatenado.csv (1200 classificacoes)

Processando bloco 3 - Arquivo: C:\Users\alexander.feitosa\Documents\Github\mada\dados\01-logs-brutos\results-prod-(2).csv
Frases no bloco 3: 150
Arquivo salvo: C:\Users\alexander.feitosa\Documents\Github\mada\dados\02-logs-processados/bloco-3-concatenado.csv (1800 classificacoes)

Processando bloco 4 - Arquivo: C:\Users\al

## 5. Consolidação das anotações e estratificação

Os dez arquivos de bloco são concatenados em `df_totais_com_duracao`. O resultado é gravado como `mqd-<N>-anotacoes-totais.csv` em `DIR_PROCESSADOS`, com `\t` como separador.

Em seguida, `GeneroCod` separa as anotações em `df_totais_masculinos` (`m`) e `df_totais_femininos` (`f`). Os caminhos resultantes ficam disponíveis em `output_masculino` e `output_feminino`. A contagem de outros valores funciona como conferência de cobertura.

In [ ]:

# Consolidação dos dez blocos processados

logs_path = DIR_PROCESSADOS

# Sequência fixa de processamento dos blocos do MQD-1222.
# Integra o protocolo experimental e assegura a reprodução
# das partições estratificadas com random_state=42.
# A ordem dos blocos não representa prioridade entre os dados.
ordem_blocos = [10, 1, 2, 3, 4, 5, 6, 7, 8, 9]
bloco_files = [
    logs_path / f"bloco-{numero}-concatenado.csv"
    for numero in ordem_blocos
]

faltantes = [str(arquivo) for arquivo in bloco_files if not arquivo.is_file()]
if faltantes:
    raise FileNotFoundError(f"Blocos não encontrados: {faltantes}")

# Confere se os dez blocos estão disponíveis
if len(bloco_files) != 10:
    raise ValueError(
        f"Esperados 10 blocos, encontrados {len(bloco_files)}."
    )

# Carrega e concatena os blocos
dfs = [
    pd.read_csv(file, sep="\t")
    for file in bloco_files
]

df_totais_com_duracao = pd.concat(
    dfs, ignore_index=True
)

# Salva o resultado consolidado
total = len(df_totais_com_duracao)
output_file = logs_path / f"mqd-{total}-anotacoes-totais.csv"

df_totais_com_duracao.to_csv(
    output_file,
    sep="\t",
    index=False
)

print(f"Arquivo criado: {output_file}")
print(f"Total de registros: {total}")
print(f"Blocos consolidados: {len(bloco_files)}")


Arquivo criado: C:\Users\alexander.feitosa\Documents\Github\mada\dados\02-logs-processados\mqd-13317-anotacoes-totais.csv
Total de registros: 13317
Blocos consolidados: 10


In [7]:

# Separa as anotações por gênero
df_totais_masculinos = df_totais_com_duracao[
    df_totais_com_duracao['GeneroCod'] == 'm'
].copy()

df_totais_femininos = df_totais_com_duracao[
    df_totais_com_duracao['GeneroCod'] == 'f'
].copy()

# Quantidades obtidas no processamento
total_m = len(df_totais_masculinos)
total_f = len(df_totais_femininos)

# Arquivos de saída em kebab-case
output_masculino = (
    DIR_PROCESSADOS / f"mqd-{total_m}-anotacoes-totais-masculinas.csv"
)

output_feminino = (
    DIR_PROCESSADOS / f"mqd-{total_f}-anotacoes-totais-femininas.csv"
)

df_totais_masculinos.to_csv(
    output_masculino, sep="\t", index=False
)

df_totais_femininos.to_csv(
    output_feminino, sep="\t", index=False
)

print(f"Masculino: {total_m} registros | {output_masculino}")
print(f"Feminino:  {total_f} registros | {output_feminino}")

# Verifica se existem anotações fora das duas categorias
total_outros = len(df_totais_com_duracao) - total_m - total_f
print(f"Outros ou não identificados: {total_outros}")


Masculino: 7015 registros | C:\Users\alexander.feitosa\Documents\Github\mada\dados\02-logs-processados\mqd-7015-anotacoes-totais-masculinas.csv
Feminino:  6302 registros | C:\Users\alexander.feitosa\Documents\Github\mada\dados\02-logs-processados\mqd-6302-anotacoes-totais-femininas.csv
Outros ou não identificados: 0


## 6. Consolidação dos rótulos por pluralidade e busca combinatória

**Entradas:** `output_masculino` e `output_feminino`. **Saída:** `mqd-<N>-majoritarias.csv` em `DIR_PROCESSADOS`.

Para cada frase e grupo de gênero, o algoritmo seleciona **quatro anotações** e define o rótulo pela **pluralidade**, isto é, pela existência de uma única classe com a maior contagem de votos. São aceitas distribuições como **4-0-0, 3-1-0 e 2-1-1**. Uma distribuição **2-2-0** permanece empatada.

**Recuperação de frases por busca combinatória.** As anotações são ordenadas pela sua posição original no arquivo (`ordem_original`). A função `selecionar_4_com_pluralidade()` percorre, em ordem lexicográfica, as combinações de quatro posições geradas por `itertools.combinations`. Se os quatro primeiros votos não definirem uma classe vencedora, a busca considera os votos excedentes, começando pelas combinações cronologicamente prioritárias. Assim, o quinto voto pode participar de uma nova seleção, e outros excedentes também podem ser considerados, sem reutilização ou duplicação de votos. A primeira combinação com pluralidade encerra a busca; **não se escolhe a combinação pela maior margem de votos**, mas pela ordem de busca.

**Descarte.** Uma frase é descartada no grupo se possuir menos de quatro anotações ou se *todas* as combinações possíveis de quatro votos persistirem sem pluralidade. O conjunto final (`df_final`) contém apenas frases com seleção válida **em ambos os grupos**. A coluna `concordancia_grupos` registra igualdade (1) ou divergência (0) dos rótulos consolidados.

*Observação:* a prioridade cronológica é operacionalizada pela **ordem dos registros de entrada**, não por uma ordenação explícita de carimbos de data e hora nesta etapa. A função `processar_grupo()` também calcula a duração média da combinação selecionada; essa variável não intervém no critério de desempate.


In [8]:
# =============================================================================
# GERACAO DO DATASET FINAL: 4 CLASSIFICACOES COM MAIORIA SIMPLES
# =============================================================================

from itertools import combinations

#df_masc_raw = pd.read_csv(logs_path / 'MQD-7015_anotacoes_totais_masculinas.csv', sep='\t')
#df_fem_raw = pd.read_csv(logs_path / 'MQD-6302_anotacoes_totais_femininas.csv', sep='\t')

df_masc_raw = pd.read_csv(output_masculino, sep='\t')
df_fem_raw = pd.read_csv(output_feminino, sep='\t')

df_masc_raw['ordem_original'] = df_masc_raw.index
df_fem_raw['ordem_original'] = df_fem_raw.index

print('=' * 80)
print('GERACAO DO DATASET FINAL')
print('=' * 80)
print(f'Anotacoes masculinas: {len(df_masc_raw)}')
print(f'Anotacoes femininas: {len(df_fem_raw)}')


def tem_pluralidade(contagem):
    valores = sorted(contagem.values, reverse=True)
    return not (len(valores) >= 2 and valores[0] == valores[1])


def selecionar_4_com_pluralidade(grupo):
    # Ordena por posicao original no arquivo (registros mais antigos primeiro).
    # Itera sobre combinacoes de 4 em ordem lexicografica, garantindo
    # que a primeira combinacao valida use os registros mais antigos possiveis.
    grupo = grupo.sort_values('ordem_original').reset_index(drop=True)
    if len(grupo) < 4:
        return None
    for idxs in combinations(range(len(grupo)), 4):
        selecao = grupo.iloc[list(idxs)]
        if tem_pluralidade(selecao['Value'].value_counts()):
            return selecao
    return None


def processar_grupo(df, genero):
    resultados = []
    descartadas = []
    for frase in df['frase'].unique():
        grupo = df[df['frase'] == frase]
        selecao = selecionar_4_com_pluralidade(grupo)
        if selecao is None:
            descartadas.append(frase)
            continue
        contagem = selecao['Value'].value_counts()
        resultados.append({
            'frase': frase,
            f'total_classificacoes_{genero}': 4,
            f'classificacao_majoritaria_{genero}': contagem.idxmax(),
            f'votos_maioria_{genero}': contagem.max(),
            f'total_positiva_{genero}': (selecao['Value'] == 'positiva').sum(),
            f'total_negativa_{genero}': (selecao['Value'] == 'negativa').sum(),
            f'total_neutra_{genero}': (selecao['Value'] == 'neutra').sum(),
            f'duracao_media_{genero}': selecao['duracao'].mean()
        })
    return pd.DataFrame(resultados), descartadas


print('\n[1] Processando anotacoes masculinas...')
df_result_masc, descartadas_masc = processar_grupo(df_masc_raw, 'masculino')
print(f'    OK {len(df_result_masc)} frases com maioria simples valida')
print(f'    X  {len(descartadas_masc)} frases descartadas (empate irresolvivel)')

print('\n[2] Processando anotacoes femininas...')
df_result_fem, descartadas_fem = processar_grupo(df_fem_raw, 'feminino')
print(f'    OK {len(df_result_fem)} frases com maioria simples valida')
print(f'    X  {len(descartadas_fem)} frases descartadas (empate irresolvivel)')

print('\n[3] Combinando datasets...')
df_final = df_result_masc.merge(df_result_fem, on='frase', how='inner')
print(f'    {len(df_final)} frases em comum (ambos generos com maioria simples valida)')

df_final['concordancia_grupos'] = (
    df_final['classificacao_majoritaria_masculino'] ==
    df_final['classificacao_majoritaria_feminino']
).astype(int)

n_concordam = df_final['concordancia_grupos'].sum()
n_discordam = len(df_final) - n_concordam

print('\n[4] Concordancia entre grupos:')
if len(df_final) > 0:
    print(f'    Concordam: {n_concordam} ({100 * n_concordam / len(df_final):.1f}%)')
    print(f'    Discordam: {n_discordam} ({100 * n_discordam / len(df_final):.1f}%)')
else:
    print('    Sem dados para comparar')

colunas_ordem = [
    'frase',
    'duracao_media_masculino', 'total_classificacoes_masculino',
    'classificacao_majoritaria_masculino', 'votos_maioria_masculino',
    'total_positiva_masculino', 'total_negativa_masculino', 'total_neutra_masculino',
    'duracao_media_feminino', 'total_classificacoes_feminino',
    'classificacao_majoritaria_feminino', 'votos_maioria_feminino',
    'total_positiva_feminino', 'total_negativa_feminino', 'total_neutra_feminino',
    'concordancia_grupos'
]
df_final = df_final[colunas_ordem]

output_file = DIR_PROCESSADOS / f'mqd-{len(df_final)}-majoritarias.csv'
df_final.to_csv(output_file, sep='\t', index=False)
print(f'\n[5] Arquivo salvo: {output_file.name}')

print('\n' + '=' * 80)
print('ESTATISTICAS DO DATASET FINAL')
print('=' * 80)
print('\nDistribuicao de votos na maioria:')
print('\nMasculino:')
print(df_final['votos_maioria_masculino'].value_counts().sort_index())
print('\nFeminino:')
print(df_final['votos_maioria_feminino'].value_counts().sort_index())
print('\nDistribuicao de classes majoritarias:')
print('\nMasculino:')
print(df_final['classificacao_majoritaria_masculino'].value_counts())
print('\nFeminino:')
print(df_final['classificacao_majoritaria_feminino'].value_counts())
print('\n' + '=' * 80)
print('DATASET FINAL GERADO COM SUCESSO')
print('=' * 80)

GERACAO DO DATASET FINAL
Anotacoes masculinas: 7015
Anotacoes femininas: 6302

[1] Processando anotacoes masculinas...
    OK 1393 frases com maioria simples valida
    X  70 frases descartadas (empate irresolvivel)

[2] Processando anotacoes femininas...
    OK 1273 frases com maioria simples valida
    X  190 frases descartadas (empate irresolvivel)

[3] Combinando datasets...
    1222 frases em comum (ambos generos com maioria simples valida)

[4] Concordancia entre grupos:
    Concordam: 1033 (84.5%)
    Discordam: 189 (15.5%)

[5] Arquivo salvo: mqd-1222-majoritarias.csv

ESTATISTICAS DO DATASET FINAL

Distribuicao de votos na maioria:

Masculino:
votos_maioria_masculino
2    129
3    469
4    624
Name: count, dtype: int64

Feminino:
votos_maioria_feminino
2    115
3    610
4    497
Name: count, dtype: int64

Distribuicao de classes majoritarias:

Masculino:
classificacao_majoritaria_masculino
positiva    481
negativa    450
neutra      291
Name: count, dtype: int64

Feminino:
cla

## 7. Registro estatístico e sensibilidade

Esta etapa produz o lastro quantitativo da consolidação e não altera o dataset de rótulos da etapa anterior. Examina: **(a)** cobertura de frases com exatamente quatro votos; **(b)** empates entre esses votos e teste z; **(c)** sensibilidade entre os quatro primeiros votos e a seleção combinatória, inclusive κ de Cohen.

**Entradas:** `output_masculino` e `output_feminino`. **Saída:** `dados/03-resultados-empiricos/registro-estatistico-consolidacao.csv`.

Os universos analisados aqui podem diferir do total de frases da base final, pois incluem verificações anteriores ao filtro de presença válida nos dois grupos.

In [9]:
# =============================================================================
# REGISTRO ESTATÍSTICO — lastro numérico da Seção IV.1.2
#   (a) cobertura de frases com exatamente quatro votos por grupo
#   (b) taxa de empate condicionada a exatamente quatro votos, com teste z
#   (c) sensibilidade: quatro primeiros votos vs. busca combinatória
# =============================================================================

from itertools import combinations
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.metrics import cohen_kappa_score

logs_path = DIR_PROCESSADOS
CLASSE_MAP = {'negativa': -1, 'neutra': 0, 'positiva': 1}

df_m = pd.read_csv(output_masculino, sep='\t')
df_f = pd.read_csv(output_feminino, sep='\t')


df_m['ordem_original'] = df_m.index
df_f['ordem_original'] = df_f.index


def ha_pluralidade(contagem):
    v = sorted(contagem.values, reverse=True)
    return not (len(v) >= 2 and v[0] == v[1])


def rotulo_primeiros_quatro(grupo):
    g = grupo.sort_values('ordem_original').head(4)
    if len(g) < 4:
        return None
    c = g['Value'].value_counts()
    return c.idxmax() if ha_pluralidade(c) else None


def rotulo_combinatorio(grupo):
    g = grupo.sort_values('ordem_original').reset_index(drop=True)
    if len(g) < 4:
        return None
    for idxs in combinations(range(len(g)), 4):
        c = g.iloc[list(idxs)]['Value'].value_counts()
        if ha_pluralidade(c):
            return c.idxmax()
    return None


def perfilar(df):
    linhas = []
    for frase, grupo in df.groupby('frase'):
        linhas.append({
            'frase': frase,
            'n_votos': len(grupo),
            'rot_p4': rotulo_primeiros_quatro(grupo),
            'rot_comb': rotulo_combinatorio(grupo),
        })
    out = pd.DataFrame(linhas)
    out['exatamente_4'] = out['n_votos'] == 4
    out['empate_p4'] = out['rot_p4'].isna()
    return out


perfil_m = perfilar(df_m)
perfil_f = perfilar(df_f)

print('=' * 78)
print('(a) COBERTURA DE VOTOS')
print('=' * 78)
for nome, p in [('masculino', perfil_m), ('feminino', perfil_f)]:
    prop = p['exatamente_4'].mean()
    print(f'  {nome:<10}: {p["exatamente_4"].sum():>5} de {len(p):>5} frases '
          f'com exatamente quatro votos ({100 * prop:.1f}%)')

print('\n' + '=' * 78)
print('(b) TAXA DE EMPATE CONDICIONADA A EXATAMENTE QUATRO VOTOS')
print('=' * 78)
x_m = int(perfil_m.loc[perfil_m['exatamente_4'], 'empate_p4'].sum())
n_m = int(perfil_m['exatamente_4'].sum())
x_f = int(perfil_f.loc[perfil_f['exatamente_4'], 'empate_p4'].sum())
n_f = int(perfil_f['exatamente_4'].sum())
p_m, p_f = x_m / n_m, x_f / n_f
p_pool = (x_m + x_f) / (n_m + n_f)
z = (p_f - p_m) / np.sqrt(p_pool * (1 - p_pool) * (1 / n_m + 1 / n_f))
p_valor = 2 * (1 - norm.cdf(abs(z)))
print(f'  masculino : {x_m}/{n_m} = {100 * p_m:.1f}%')
print(f'  feminino  : {x_f}/{n_f} = {100 * p_f:.1f}%')
print(f'  z = {z:.3f} | p = {p_valor:.4g} (bilateral, proporções independentes)')

print('\n' + '=' * 78)
print('(c) SENSIBILIDADE: QUATRO PRIMEIROS VOTOS vs. BUSCA COMBINATÓRIA')
print('=' * 78)
for nome, p in [('masculino', perfil_m), ('feminino', perfil_f)]:
    validos_p4 = p['rot_p4'].notna().sum()
    validos_comb = p['rot_comb'].notna().sum()
    ambos = p[p['rot_p4'].notna() & p['rot_comb'].notna()]
    identicos = (ambos['rot_p4'] == ambos['rot_comb']).sum()
    print(f'  {nome:<10}: p4 valida {validos_p4} | combinatoria valida {validos_comb} '
          f'| recuperadas {validos_comb - validos_p4}')
    print(f'  {"":<10}  rotulos identicos quando ambos validos: '
          f'{identicos} de {len(ambos)}')

par_p4 = (perfil_m[['frase', 'rot_p4']].rename(columns={'rot_p4': 'm'})
          .merge(perfil_f[['frase', 'rot_p4']].rename(columns={'rot_p4': 'f'}), on='frase')
          .dropna())
par_comb = (perfil_m[['frase', 'rot_comb']].rename(columns={'rot_comb': 'm'})
            .merge(perfil_f[['frase', 'rot_comb']].rename(columns={'rot_comb': 'f'}), on='frase')
            .dropna())
k_p4 = cohen_kappa_score(par_p4['m'].map(CLASSE_MAP), par_p4['f'].map(CLASSE_MAP))
k_comb = cohen_kappa_score(par_comb['m'].map(CLASSE_MAP), par_comb['f'].map(CLASSE_MAP))
print(f'\n  kappa_anot (quatro primeiros votos) : {k_p4:.4f}  (N = {len(par_p4)})')
print(f'  kappa_anot (busca combinatoria)    : {k_comb:.4f}  (N = {len(par_comb)})')

registro = pd.DataFrame([
    {'metrica': 'prop_exatamente_4_masculino', 'valor': perfil_m['exatamente_4'].mean()},
    {'metrica': 'prop_exatamente_4_feminino',  'valor': perfil_f['exatamente_4'].mean()},
    {'metrica': 'taxa_empate_4votos_masculino', 'valor': p_m},
    {'metrica': 'taxa_empate_4votos_feminino',  'valor': p_f},
    {'metrica': 'z_stat_empate', 'valor': z},
    {'metrica': 'p_valor_empate', 'valor': p_valor},
    {'metrica': 'kappa_anot_primeiros4', 'valor': k_p4},
    {'metrica': 'N_primeiros4', 'valor': len(par_p4)},
    {'metrica': 'kappa_anot_combinatorio', 'valor': k_comb},
    {'metrica': 'N_combinatorio', 'valor': len(par_comb)},
])
destino = DIR_DADOS / "03-resultados-empiricos"
destino.mkdir(parents=True, exist_ok=True)

arquivo_registro = destino / "registro-estatistico-consolidacao.csv"

registro.to_csv(arquivo_registro, index=False)
print(f"\n[OK] Salvo: {arquivo_registro}")

(a) COBERTURA DE VOTOS
  masculino :   600 de  1463 frases com exatamente quatro votos (41.0%)
  feminino  :  1163 de  1463 frases com exatamente quatro votos (79.5%)

(b) TAXA DE EMPATE CONDICIONADA A EXATAMENTE QUATRO VOTOS
  masculino : 70/600 = 11.7%
  feminino  : 190/1163 = 16.3%
  z = 2.620 | p = 0.008781 (bilateral, proporções independentes)

(c) SENSIBILIDADE: QUATRO PRIMEIROS VOTOS vs. BUSCA COMBINATÓRIA
  masculino : p4 valida 1307 | combinatoria valida 1393 | recuperadas 86
              rotulos identicos quando ambos validos: 1307 de 1307
  feminino  : p4 valida 1226 | combinatoria valida 1273 | recuperadas 47
              rotulos identicos quando ambos validos: 1226 de 1226

  kappa_anot (quatro primeiros votos) : 0.7936  (N = 1111)
  kappa_anot (busca combinatoria)    : 0.7664  (N = 1222)

[OK] Salvo: C:\Users\alexander.feitosa\Documents\Github\mada\dados\03-resultados-empiricos\registro-estatistico-consolidacao.csv
